Note: this nootbook if for preforming BERTTOpic clustering when the data is too much big to fit in memory leading to Oout of memory (OOM). The nootbook dideivde the data to chunks and  fit a sepreate berttopic model on eeach of the cunnkks. 

# Imports

In [ ]:
# ML
import pandas as pd
import numpy as np
import pyarrow  # ensures pandas can read/write parquet
import math

# Models
from bertopic import BERTopic
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

# System
import gc
from pathlib import Path
from datetime import datetime
# import subprocess
import sys
import fcntl
import tempfile
import os



# Pyton
from functools import wraps
from typing import Iterator, List, Tuple

# Visualization
import matplotlib.pyplot as plt
from IPython.display import display
from pprint import pprint


# Loading and Exporting


In [ ]:
dataset_name = "Labeled_Datasets/Ecuador"  # folder path
raw_file_name = "Ecuador_part_all.gzip.parquet"  # file name
date = "2026_01_10"

# For testing
# raw_file_name = "Ecuador_part_1.gzip.parquet"  # file name
# date = "2025_12_14"

In [ ]:
raw_file_name_no_suffixes = raw_file_name.split(".")[0]
processed_file_name = f"Topic_Ner_{raw_file_name}" # the outpput of clean_translate_topics_NER.ipynb

folder_path = Path("../results") / dataset_name / raw_file_name_no_suffixes / date
processed_df_path = folder_path / processed_file_name


df_output_path = folder_path / f"processed_{raw_file_name_no_suffixes}.gzip.parquet"


print(f"folder_path: {folder_path}")
print(f"processed_df_path: {processed_df_path}")
print(f"df_output_path: {df_output_path}")

In [ ]:
df = pd.read_parquet(processed_df_path)

In [ ]:
print(f"df shape: {df.shape}")
print("df columns:")
pprint(list(df.columns))

In [ ]:
display(df.head())

# Utils

## Parameters

In [ ]:
# min_topic_size = int(min(len(df) / 10, int(math.log(100, 1.1)))) # 100 -> 10, 50K -> 113, 1M -> 144
min_topic_size = 2**8 # 256
chunk_size = 2**17 # 131,072

## Logs

In [ ]:
progress_reports_folder = folder_path / "progress_reports"
progress_reports_folder.mkdir(exist_ok=True)
REPORT_PATH = progress_reports_folder / f"progress_report_min_topic_size_{min_topic_size}.txt"

def write_report(msg: str):
    timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    line = f"{timestamp} | {msg}"

    print(line)
    with open(REPORT_PATH, "a") as f:
        f.write(line + "\n")


def report_done(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        try:
            result = func(*args, **kwargs)
            write_report(f"Completed: {func.__name__}")
            return result
        except Exception as e:
            write_report(f"FAILED: {func.__name__} | {e}")
            raise
    return wrapper

In [ ]:
write_report(" ############# running: big_data_topic_clustering.ipynb ############# ")
write_report(f"Today date: {datetime.now().strftime('%Y-%m-%d')}")
write_report(f"Loaded df: {processed_file_name}.")
write_report(f"df shape: {df.shape}")
write_report(f"min_topic_size: {min_topic_size}")
print("df head:")
display(df.head(3))

## Save/Update df

In [ ]:
# @report_done
# def save_df(
#         df: pd.DataFrame,
#         df_output_path: Path = df_output_path
#     )-> None :
#     df.to_parquet(df_output_path, index=False, compression="gzip")
#     write_report(f"df is saved to: {df_output_path}")

In [ ]:
from __future__ import annotations

@report_done
def update_df_collumns_atomically(
    df: pd.DataFrame,
    collums_to_update: str | list[str],
    df_output_path: Path = df_output_path,
    compression: str = "gzip",
) -> pd.DataFrame:
    """Safely update or add columns in a shared Parquet file.
    This assumes the same row order/length in df and df_from_file. 

    Performs a transaction-style update:
    exclusive lock -> read latest -> update columns -> write temp -> os.replace.

    Args:
        df: DataFrame containing the new column values.
        collums_to_update: Column name or list of column names to update/add.
        folder_path: Directory containing the Parquet file.
        file_name: Name of the Parquet file to update.
        compression: Parquet compression codec. Default is "gzip".

    Returns:
        The updated DataFrame that was written to disk.

    Raises:
        RuntimeError: If reading, updating, or writing fails. Original exception
            is attached as context.
    """
    lock_path = df_output_path.with_suffix(df_output_path.suffix + ".lock")

    if isinstance(collums_to_update, str):
        collums_to_update = [collums_to_update]

    tmp_path: Path | None = None

    with open(lock_path, "w") as lockf:
        fcntl.flock(lockf, fcntl.LOCK_EX)
        try:
            if not df_output_path.exists():
                df_from_file = df.copy()
            else:
                df_from_file = pd.read_parquet(df_output_path)

            for col in collums_to_update:
                df_from_file[col] = df[col]

            fd, tmp = tempfile.mkstemp(
                dir=str(df_output_path.parent),
                prefix=df_output_path.name + ".",
                suffix=".tmp",
            )
            os.close(fd)
            tmp_path = Path(tmp)

            df_from_file.to_parquet(tmp_path, index=False, compression=compression)
            os.replace(tmp_path, df_output_path)

            write_report("[update_df_collumns_atomically]\n Updated columns: " + ", ".join(collums_to_update) + f" \nsave to {df_output_path}")

            return df_from_file

        except Exception as e:
            raise RuntimeError(f"Failed to update parquet file: {df_output_path}") from e

        finally:
            if tmp_path is not None and tmp_path.exists():
                tmp_path.unlink(missing_ok=True)
            fcntl.flock(lockf, fcntl.LOCK_UN)

# English Topic Clustering

Note: the topic clustering doent influenced by the entities since it is preformed on the text without the entitites.

## Utils

### Posts Topic Stats

In [ ]:
@report_done
def make_topic_stats(df, topic_col_name):
    """
    Calculate per-topic post counts.

    Args:
        df (pd.DataFrame): Data with topics and 'is_control'.
        topic_col_name (str): Column name of topic labels.

    Returns:
        topic_stats (pd.DataFrame): Colums: [topic, total_count, io_Count, legitimate_posts, io_percent].
    """
    topic_stats = (
        df.groupby(topic_col_name)
          .agg(
              total_count=(topic_col_name, "count"),
              io_count=("is_control", lambda x: (~x).sum())
          )
          .reset_index()
    )
    topic_stats["legitimate_posts"] = topic_stats["total_count"] - topic_stats["io_count"]
    topic_stats["io_percent"] = topic_stats["io_count"] / topic_stats["total_count"] * 100
    return topic_stats

In [ ]:
from matplotlib.lines import Line2D

def plot_topic_stats(topic_stats, topic_col_name, output_folder_path: str | Path = ""):
    """
    Plot stacked IO vs legitimate post counts per topic.
    Saves the fig to output_path if provided.

    Args:
        topic_stats (pd.DataFrame): Output of make_topic_stats().
        topic_col_name (str): Topic column to plot.
        output_folder_path (str): Path to *folder* save the plot.

    Returns:
        None
    """
    topics = topic_stats[topic_col_name]
    io_count = topic_stats["io_count"]
    legitimate_count = topic_stats["legitimate_posts"]
    io_percent = topic_stats["io_percent"]

    plt.figure(figsize=(12, 6))

    # 🔴 Bottom: IO posts
    bar_io = plt.bar(topics, io_count, label="IO posts", color="red")

    # 🔵 Top: Non-IO posts stacked above IO
    bar_legit = plt.bar(topics, legitimate_count, bottom=io_count, label="Non-IO posts")

    # Add IO percentage text
    ax = plt.gca()
    for x, io, legit, pct in zip(topics, io_count, legitimate_count, io_percent):
        total_height = io + legit
        ax.text(
            x,
            total_height + 0.5,              # slightly above the bar
            f"{pct:.1f}%",                 # format like 23.5%
            ha="center",
            va="bottom",
            fontsize=8,
            fontweight="regular"
        )

    plt.title(f"Post Count per Topic ({topic_col_name})")
    plt.xlabel("Topic ID")
    plt.ylabel("Number of Posts")
    plt.xticks(topics)
    plt.grid(axis="y")

    percent_legend = Line2D([], [], color="none")
    total_posts_legend = Line2D([], [], color="none")
    plt.legend(
        handles=[bar_io, bar_legit, percent_legend, total_posts_legend],
        labels=[
            "IO posts",
            "Non-IO posts",
            "% IO Posts Ratio",
            f"Total posts: {topic_stats['total_count'].sum()}",
        ],
        loc="best",
    )


    plt.tight_layout()


    if output_folder_path:
        output_folder_path = Path(output_folder_path)
        plot_file_path = output_folder_path / (f"{topic_col_name}_topic_clustering.png")
        plt.savefig(plot_file_path, dpi=300)
        print(f"plot is saved to {plot_file_path}")

    plt.show()

    plt.close()

### Accoutns Topic Stats

In [ ]:
@report_done
def make_topic_account_stats(
    df: pd.DataFrame,
    topic_col: str,
    author_col: str = "accountid",
    is_control_col: str = "is_control",
    threshold: float = 0.5,
) -> pd.DataFrame:
    """Compute per-topic unique account counts (IO vs control) using an author-level threshold.

    Args:
        df: Post-level DataFrame.
        topic_col: Topic column name.
        author_col: Author identifier column.
        is_control_col: Column where 1 = control, 0 = IO.
        threshold: IO-rate threshold to label an author as IO (1 = IO).

    Returns:
        DataFrame with per-topic counts of unique IO/control accounts and IO percentage.
    """
    # Author-level IO rate: 1 - mean(is_control)
    author_io_rate = 1 - df.groupby(author_col)[is_control_col].mean()
    author_io_tag = (author_io_rate >= threshold).astype(int)

    topic_author_df = df[[topic_col, author_col]].drop_duplicates().copy()
    topic_author_df["author_io_tag"] = topic_author_df[author_col].map(author_io_tag)

    stats = (
        topic_author_df
        .groupby(topic_col)["author_io_tag"]
        .value_counts()
        .unstack(fill_value=0)
    )

    # Ensure both columns exist even if absent in data
    stats = stats.reindex(columns=[0, 1], fill_value=0).rename(
        columns={0: "control_accounts", 1: "io_accounts"}
    ).reset_index()

    stats["total_accounts"] = stats["control_accounts"] + stats["io_accounts"]
    stats["io_percent"] = (stats["io_accounts"] / stats["total_accounts"]).mul(100)

    return stats

In [ ]:
def plot_topic_account_stats(
    topic_account_stats: pd.DataFrame,
    topic_col_name: str,
    output_folder_path: Path | str | None = None,
):
    """Plot IO vs control account counts per topic.

    Args:
        topic_account_stats: Output of make_topic_account_stats().
        topic_col_name: Topic column name.
        output_folder_path: Optional folder to save the plot.

    Returns:
        None
    """
    topics = topic_account_stats[topic_col_name]
    io_count = topic_account_stats["io_accounts"]
    control_count = topic_account_stats["control_accounts"]
    io_percent = topic_account_stats["io_percent"]

    plt.figure(figsize=(12, 6))

    # 🔴 Bottom: IO accounts
    bar_io = plt.bar(topics, io_count, label="IO accounts", color="red")

    # 🔵 Top: Control accounts
    bar_control = plt.bar(
        topics,
        control_count,
        bottom=io_count,
        label="Control accounts",
    )

    # Add IO percentage text
    ax = plt.gca()
    for x, io, ctrl, pct in zip(topics, io_count, control_count, io_percent):
        total = io + ctrl
        ax.text(
            x,
            total + 0.3,
            f"{pct:.1f}%",
            ha="center",
            va="bottom",
            fontsize=8,
            fontweight="regular",
        )

    plt.title(f"Unique Accounts per Topic ({topic_col_name})")
    plt.xlabel("Topic ID")
    plt.ylabel("Number of Unique Accounts")
    plt.xticks(topics)
    plt.grid(axis="y")

    # Legend with extra info lines
    percent_legend = Line2D([], [], color="none")
    total_accounts_legend = Line2D([], [], color="none")

    plt.legend(
        handles=[bar_io, bar_control, percent_legend, total_accounts_legend],
        labels=[
            "IO accounts",
            "Control accounts",
            "% IO Accounts Ratio",
            f"Total accounts: {topic_account_stats['total_accounts'].sum()}",
        ],
        loc="best",
    )

    if output_folder_path:
        output_folder_path = Path(output_folder_path)
        plot_file_path = output_folder_path / f"{topic_col_name}_topic_accounts.png"
        plt.savefig(plot_file_path, dpi=300)
        print(f"Plot saved to {plot_file_path}")

    plt.show()
    plt.close()

## Embeddings

In [ ]:
from sentence_transformers import SentenceTransformer

post_text_col = "translated_post"

posts_texts = df[post_text_col].astype(str).tolist()

model = SentenceTransformer("digio/Twitter4SSE")

embeddings = model.encode(
    posts_texts,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True, 
)

embeddings = embeddings.astype("float32", copy=False) # in order to use less memory
write_report(f"Generated embeddings shape: {embeddings.shape}")

## BERTopic

[BERTopic Parametes](https://maartengr.github.io/BERTopic/getting_started/parameter%20tuning/parametertuning.html#calculate_probabilities)

In [ ]:
# min topic size are define in the stat of the nootbook

BERTopic_topics_col_name = f"BERTopic_topic_{min_topic_size}"
BERTopic_probs_col_name = f"BERTopic_prob_{min_topic_size}"

### Build Chunks Models

https://maartengr.github.io/BERTopic/getting_started/merge/merge.html

In [ ]:
# Make output folder for chunk models
chunk_models_output_folder_path = Path(folder_path) / f"BERTopic_models_min_topic_size_{min_topic_size}"
chunk_models_output_folder_path.mkdir(exist_ok=True)

In [ ]:
def iter_text_chunks(
    texts: List[str],
    chunk_size: int,
) -> Iterator[Tuple[int, List[str]]]:
    """Yield (start_index, text_chunk) pairs of fixed size.

    Args:
        texts: List of documents.
        chunk_size: Max number of documents per chunk.

    Yields:
        Tuple[int, List[str]]: (start_index, chunk_texts).
    """
    for start in range(0, len(texts), chunk_size):
        yield start, texts[start : start + chunk_size]


@report_done
def build_and_save_chunk_models(
    texts: List[str],
    embeddings: np.ndarray,
    chunk_size: int,
    min_topic_size: int,
    output_dir: str | Path,
) -> None:
    """Train BERTopic models on chunks and save each to disk.

    Args:
        texts: Documents.
        embeddings: Precomputed embeddings (n_docs, dim).
        chunk_size: Max docs per chunk.
        min_topic_size: BERTopic min_topic_size.
        output_dir: Directory to save chunk models.
    """
    
    for idx, (start, text_chunk) in enumerate(iter_text_chunks(texts, chunk_size)):
        end = start + len(text_chunk)
        emb_chunk = embeddings[start:end]

        model = BERTopic(
            language="english",
            min_topic_size=min_topic_size,
            low_memory=True,
            calculate_probabilities=False,
        )

        model.fit(text_chunk, emb_chunk)

        model_path = output_dir / f"chunk_model_{idx}"
        model.save(model_path)
        write_report(f"Saved chunk model {idx} for texts {start} to {end} at {model_path}")

        # free RAM
        del model, emb_chunk, text_chunk # delete references
        gc.collect() # Garbage Collector

### Merge Chunks Models

In [ ]:
@report_done
def load_and_merge_BERTopic_models(
    chunk_models_dir: str | Path,
    min_similarity: float = 0.7, # default value is 0.7 in BERTopic docs
) -> BERTopic:
  
    chunk_models_dir = Path(chunk_models_dir)
    paths = [p for p in chunk_models_dir.iterdir()]

    if not paths:
        raise ValueError(f"No chunk models found in {chunk_models_dir}")
    
    merged_model  = None

    for p in paths:
        chunk_model = BERTopic.load(p)

        if merged_model is None: # first model
            merged_model = chunk_model
        else:
            merged_model = BERTopic.merge_models([merged_model, chunk_model], min_similarity=min_similarity)
            del chunk_model
            gc.collect()


    if merged_model is None:
        raise RuntimeError("merged_model is None after loading chunk models.")
    

    # save merged model
    merge_model_file_name = f"merged_BERTopic_model_min_similarity_{min_similarity}"
    merged_model.save(chunk_models_dir / merge_model_file_name)
    write_report(f"Merged model saved to {chunk_models_dir / merge_model_file_name}")

    return merged_model

### Run BERTopic Modeling

In [ ]:
build_and_save_chunk_models(
    texts=posts_texts,
    embeddings=embeddings,
    chunk_size=chunk_size,
    min_topic_size=min_topic_size,
    output_dir=chunk_models_output_folder_path,
)

merge_BERTopic = load_and_merge_BERTopic_models(chunk_models_dir = chunk_models_output_folder_path)

# upodate df
BERTopic_topic, BERT_probs = merge_BERTopic.transform(posts_texts, embeddings)
df[BERTopic_topics_col_name] = BERTopic_topic
df[BERTopic_probs_col_name] = BERT_probs

### Update and Save df to Disk

Using file for lock to enable parallel instances.

In [ ]:
collum_to_update = [BERTopic_topics_col_name, BERTopic_probs_col_name]

df = update_df_collumns_atomically(df, collum_to_update)

write_report("Saved df with BERTopic topics.")

### Graph and Statstistics

In [ ]:
try:
    fig = merge_BERTopic.visualize_topics(custom_labels=True)
    display(fig)   # show it
except Exception as e:
    print("Not enough topics for visualization")
    print("Error:", e)

In [ ]:
merge_BERTopic.get_topic_info()

Topic Post Statstistics

In [ ]:
BERT_topic_stats = make_topic_stats(df, BERTopic_topics_col_name)
plot_topic_stats(BERT_topic_stats, BERTopic_topics_col_name, folder_path)

Topic Account Statstistics

In [ ]:
topic_account_stats = make_topic_account_stats(df, BERTopic_topics_col_name)
display(topic_account_stats)
plot_topic_account_stats(topic_account_stats, BERTopic_topics_col_name, folder_path)

# Stop before K-Means

In [ ]:
# write_report("BERTopic Finished")
# write_report("Dont continue for k-means")
# sys.exit(0)

## K-Means

K Means is done by the `translate_topics_entities.ipynb`

In [ ]:
# kmeans_topic_col_name = "K_Means_topic" # k value will be added to the collom

### Run K-Means

In [ ]:
# @report_done
# def auto_kmeans_topics(embeddings, k_min=2**2, k_max=2**7, step = 4):
#     best_k = None
#     best_score = -1
#     best_labels = None

#     # print memory summary
#     # write_report(f"auto_kmeans_topics: \n {subprocess.check_output('nvidia-smi', shell=True).decode()}")

#     for k in range(k_min, k_max + 1, step):
#         print(f"Trying k={k}")
#         km = KMeans(n_clusters=k, random_state=42, n_init="auto")
#         labels = km.fit_predict(embeddings)
#         score = silhouette_score(embeddings, labels)

#         if score > best_score:
#             best_score = score
#             best_k = k
#             best_labels = labels

#     write_report(f"Best k for K-Means: {best_k} with silhouette score: {best_score:.4f}")

#     return best_labels, best_k

In [ ]:
# labels, best_k = auto_kmeans_topics(embeddings, k_min=2**2, k_max=2**7)
# kmeans_topic_col_name = f"K_Means_topic_{best_k}"
# df[kmeans_topic_col_name] = labels
# write_report(f"k-means finished, best k {best_k}, topics saved to df in the column {kmeans_topic_col_name}")
# df.head()

### Graph and Statstistics

Topic Post Statstistics

In [ ]:
# topic_stats = make_topic_stats(df, kmeans_topic_col_name)
# plot_topic_stats(topic_stats, kmeans_topic_col_name, folder_path)

Topic Account Statstistics

In [ ]:
# topic_account_stats = make_topic_account_stats(df, kmeans_topic_col_name)
# plot_topic_account_stats(topic_account_stats, kmeans_topic_col_name, folder_path)

### Save Model and df

In [ ]:
# update_df_collumns_atomically(df, kmeans_topic_col_name)
# write_report("Saved df with BERTopic topics.")